In [1]:
# InGame Score Prediction Model Notebook 1: Live Data Retrieval & Raw Data Acquisition

In [3]:
# Header For Imports

import sys
import os

# Determine the project root (assuming structure: score-genius/ with backend/ and notebooks/ as siblings)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Notebook 1: Live Data Retrieval
import pandas as pd
import pytz
from datetime import datetime
import traceback
from caching.supabase_client import supabase

from backend.models.features import NBAFeatureGenerator
from backend.utils.analysis_utils import convert_time_to_minutes, ensure_numeric_features


In [4]:
# Cell 1: Imports, Live Game Data Retrieval in Pacific Time

import pandas as pd
import pytz
from datetime import datetime
import traceback
from caching.supabase_client import supabase

def fetch_live_games_in_pacific_time():
    """
    Fetch actively live games from 'nba_live_game_stats' using the stored 'status'
    field. A game is considered live if its status is one of:
      {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}
    """
    live_statuses = {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}
    
    try:
        response = supabase.table("nba_live_game_stats").select("*").execute()
        if not response.data:
            print("No data found in 'nba_live_game_stats' table.")
            return pd.DataFrame()
        
        df = pd.DataFrame(response.data)
        
        # Ensure 'game_date' exists
        if "game_date" not in df.columns:
            print("Column 'game_date' not found!")
            return pd.DataFrame()
        
        # Convert game_date (stored as fake UTC) to datetime and then to Pacific Time
        df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce", utc=True)
        pacific_tz = pytz.timezone("America/Los_Angeles")
        df["game_date_pt"] = df["game_date"].dt.tz_convert(pacific_tz)
        df["date_only_pt"] = df["game_date_pt"].dt.date
        
        # Filter for games scheduled for today in Pacific Time
        now_pt = datetime.now(pacific_tz)
        today_pt = now_pt.date()
        df_today = df[df["date_only_pt"] == today_pt].copy()
        if df_today.empty:
            print("No games match today's date in Pacific Time.")
            return pd.DataFrame()
        
        # Make sure the status field is available and in uppercase
        if "status" not in df_today.columns:
            print("Column 'status' not found!")
            return pd.DataFrame()
        df_today["status"] = df_today["status"].astype(str).str.upper()
        
        # Filter only active games (using the stored status)
        active_games = df_today[df_today["status"].isin(live_statuses)].copy()
        
        print(f"ACTIVELY LIVE GAMES: {len(active_games)}")
        if not active_games.empty:
            for _, row in active_games.iterrows():
                print(
                    f"{row.get('home_team', 'Unknown')} vs {row.get('away_team', 'Unknown')}: "
                    f"{row.get('home_score', 'N/A')}-{row.get('away_score', 'N/A')}, "
                    f"Status: {row.get('status')}"
                )
        else:
            print("No active games found!")
        
        return active_games
        
    except Exception as e:
        print(f"Error fetching live games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

# Test the function
live_games = fetch_live_games_in_pacific_time()
if not live_games.empty:
    print("Successfully retrieved active games (Pacific Time).")
else:
    print("No active game data available.")

No data found in 'nba_live_game_stats' table.
No active game data available.


In [10]:
# Cell 3: Improved Live Game Data Retrieval with Clear Function Separation 
import pandas as pd
import pytz
import time
import traceback
import sys
import os

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import from the correct module path
from backend.utils.analysis_utils import convert_time_to_minutes

def fetch_pacific_time():
    """Gets current time in Pacific timezone"""
    pacific_tz = pytz.timezone("America/Los_Angeles")
    return datetime.now(pacific_tz)

def get_active_live_games():
    """
    Fetch actively live games from the NBA database with improved error handling.
    
    Returns:
        DataFrame: Live game data or empty DataFrame if no games found
    """
    max_retries = 3
    retry_delay = 2 # seconds
    
    for attempt in range(max_retries):
        try:
            print(f"Attempting to fetch live game data (attempt {attempt+1}/{max_retries})...")
            response = supabase.table("nba_live_game_stats").select("*").execute()
            raw_data = response.data
            
            if not raw_data:
                print("No live game data available.")
                return pd.DataFrame()
            
            # Convert to DataFrame
            raw_df = pd.DataFrame(raw_data)
            print(f"Retrieved {len(raw_df)} games from nba_live_game_stats")
            
            # Process minutes if present
            if 'minutes' in raw_df.columns:
                raw_df['minutes_numeric'] = raw_df['minutes'].apply(convert_time_to_minutes)
                raw_df.drop(columns=['minutes'], inplace=True)
            
            # Convert game_date to datetime and filter for today's games in Pacific Time
            if "game_date" in raw_df.columns:
                raw_df["game_date"] = pd.to_datetime(raw_df["game_date"], errors="coerce", utc=True)
                pacific_tz = pytz.timezone("America/Los_Angeles")
                raw_df["game_date_pt"] = raw_df["game_date"].dt.tz_convert(pacific_tz)
                raw_df["date_only_pt"] = raw_df["game_date_pt"].dt.date
                
                now_pt = fetch_pacific_time()
                today_pt = now_pt.date()
                raw_df = raw_df[raw_df["date_only_pt"] == today_pt].copy()
                print(f"Found {len(raw_df)} games scheduled for today")
            
            # Ensure the 'status' column exists and convert to uppercase
            if "status" not in raw_df.columns:
                print("Column 'status' not found!")
                return pd.DataFrame()
                
            raw_df["status"] = raw_df["status"].astype(str).str.upper()
            
            # Mark active games based solely on the status field
            live_statuses = {"Q1", "Q2", "Q3", "Q4", "OT", "BT", "HT"}
            raw_df['is_active_now'] = raw_df["status"].isin(live_statuses)
            active_df = raw_df[raw_df['is_active_now']].copy()
            print(f"Found {len(active_df)} actively live games")
            
            if active_df.empty:
                print("No actively live games found.")
                return pd.DataFrame()
            
            # Optionally, check for missing values in critical columns
            critical_cols = ['home_q1', 'home_q2', 'home_q3', 'home_q4',
                            'away_q1', 'away_q2', 'away_q3', 'away_q4',
                            'home_score', 'away_score', 'current_quarter']
            missing_data = {}
            
            for col in critical_cols:
                if col in active_df.columns:
                    missing_count = active_df[col].isna().sum()
                    if missing_count > 0:
                        missing_data[col] = missing_count
            
            if missing_data:
                print("\nWARNING: Missing values detected in critical columns:")
                for col, count in missing_data.items():
                    print(f" • {col}: {count} missing values ({count/len(active_df)*100:.1f}%)")
            
            return active_df
            
        except Exception as e:
            print(f"Connection error: {e}")
            traceback.print_exc()
            
            if attempt < max_retries - 1:
                print(f"Retrying in {retry_delay} seconds...")
                time.sleep(retry_delay)
                retry_delay *= 2
            else:
                print("Maximum retries reached. Please check your network connection and Supabase configuration.")
                return pd.DataFrame()

# Get actively live games
raw_df = get_active_live_games()

# Display the results
if not raw_df.empty:
    print("\nLatest Active Game Data:")
    display(raw_df)
else:
    print("\nNo actively live game data available for analysis.")

Attempting to fetch live game data (attempt 1/3)...
Retrieved 11 games from nba_live_game_stats
Found 11 games scheduled for today
Found 3 actively live games

Latest Active Game Data:


,id,game_id,home_team,away_team,home_score,away_score,home_q1,home_q2,home_q3,home_q4,...,home_fg_attempted,away_fg_made,away_fg_attempted,home_ft_made,home_ft_attempted,away_ft_made,away_ft_attempted,game_date_pt,date_only_pt,is_active_now
0,419,414871,Indiana Pacers,Dallas Mavericks,48.0,35.0,33,15,0,0,...,29,13,26,9,10,4,4,2025-03-19 09:00:00-07:00,2025-03-19,True
1,420,414872,Orlando Magic,Houston Rockets,41.0,34.0,23,18,0,0,...,35,8,27,4,4,12,12,2025-03-19 09:00:00-07:00,2025-03-19,True
2,421,414873,Miami Heat,Detroit Pistons,7.0,8.0,7,0,0,0,...,5,4,8,1,1,0,0,2025-03-19 09:30:00-07:00,2025-03-19,True


In [12]:
# Cell 4: Save Data for Next Notebook in Pipeline

import json  # Add this import
import os
import traceback
from datetime import datetime

def save_data_for_pipeline(live_data_df, output_file='data/live_game_data.pkl'):
    """
    Save processed live game data for use in the next notebook in the pipeline.
    
    Args:
        live_data_df: DataFrame with live game data
        output_file: File to save data to
    """
    if live_data_df.empty:
        print("No live game data to save. Pipeline will use cached data if available.")
        return False
    
    try:
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        
        # Save the DataFrame
        live_data_df.to_pickle(output_file)
        print(f"Live game data saved to {output_file} for pipeline processing")
        
        # Also save metadata about the data
        metadata = {
            'timestamp': datetime.now().isoformat(),
            'num_games': len(live_data_df),
            'columns': list(live_data_df.columns),
            'active_games': int(live_data_df['is_active_now'].sum()) if 'is_active_now' in live_data_df.columns else 0,
            'statuses': live_data_df['status'].unique().tolist() if 'status' in live_data_df.columns else []
        }
        
        with open(output_file.replace('.pkl', '_metadata.json'), 'w') as f:
            json.dump(metadata, f)
        
        return True
    
    except Exception as e:
        print(f"Error saving live game data: {e}")
        traceback.print_exc()
        return False

# Save the active games data for the next stage in the pipeline
if not raw_df.empty:
    save_data_for_pipeline(raw_df)
    print("Data is ready for preprocessing in Notebook 2")
else:
    print("No live game data available. Pipeline cannot continue to prediction.")

Live game data saved to data/live_game_data.pkl for pipeline processing
Data is ready for preprocessing in Notebook 2
